In [1]:
from monte import train_with_cv, fine_tune_with_cv, Monte
import pandas as pd

load data

training set

In [2]:
df_beta_train = pd.read_parquet("../../data/methylation/train_pan-cancer_beta.parquet")
df_meta_train = pd.read_csv("../../data/methylation/train_pan-cancer_meta.csv")
df_meta_train = df_meta_train.set_index("Barcode", drop=False)
df_meta_train = df_meta_train.loc[df_beta_train.index]

In [3]:
cancer_types = df_meta_train["Cancer.type"].unique()
target_metric = "CPE"

testing set

In [4]:
df_beta_test = pd.read_parquet("../../data/methylation/test_pan-cancer_beta.parquet")
df_meta_test = pd.read_csv("../../data/methylation/test_pan-cancer_meta.csv")
df_meta_test = df_meta_test.set_index("Barcode", drop=False)
df_meta_test = df_meta_test.loc[df_beta_test.index]

probes on sex chromosomes

In [5]:
xy_probes = pd.read_csv('/grain/mk98/cancer-methyl/probe_selection_files/xy_probes.txt', header=None)[0].tolist()

cancer model training

In [6]:
cancer_need_checked = ["KIRC", "KIRP", "KICH", "GBM", "LGG", "UCEC", "UCS", "LUAD", "LUSC", "COAD", "READ"]

cancer_from_same_tissue = {
    "KIRC": ["KIRC", "KIRP", "KICH"],
    "KIRP": ["KIRC", "KIRP", "KICH"],
    "KICH": ["KIRC", "KIRP", "KICH"],
    "GBM": ["GBM", "LGG"],
    "LGG": ["GBM", "LGG"],
    "UCEC": ["UCEC", "UCS"],
    "UCS": ["UCEC", "UCS"],
    "LUAD": ["LUAD", "LUSC"],
    "LUSC": ["LUAD", "LUSC"],
    "COAD": ["COAD", "READ"],
    "READ": ["COAD", "READ"]
}

In [10]:
data_model_stats = []

for cancer in cancer_types:
    # owing to OV has only 4 samples
    if cancer == "OV":
        continue

    # for pan-cancer model training
    if cancer in cancer_need_checked:
        df_meta_pan = df_meta_train[df_meta_train["Cancer.type"].isin(cancer_from_same_tissue[cancer]) == False]
        df_beta_pan = df_beta_train[df_meta_train["Cancer.type"].isin(cancer_from_same_tissue[cancer]) == False]
    else:
        df_meta_pan = df_meta_train[df_meta_train["Cancer.type"] != cancer]
        df_beta_pan = df_beta_train[df_meta_train["Cancer.type"] != cancer]
    
    # for bayesian transfer learning
    df_meta_cancer = df_meta_train[df_meta_train["Cancer.type"] == cancer]
    df_beta_cancer = df_beta_train[df_meta_train["Cancer.type"] == cancer]

    print(f"{cancer}: {df_meta_train.shape[0]} = {df_meta_pan.shape[0]} + {df_meta_cancer.shape[0]} : {df_meta_train.shape[0] == df_meta_pan.shape[0] + df_meta_cancer.shape[0]}")
    
    # pan-cancer
    pan_model = train_with_cv(df_beta_pan, df_meta_pan[target_metric])

    # cancer-specific
    cancer_model = fine_tune_with_cv(pan_model, df_beta_cancer, df_meta_cancer[target_metric])

    data_model_stats.append([cancer, df_beta_pan.shape[0], df_beta_cancer.shape[0], pan_model.best_top_n, cancer_model.best_tau2])
    # cancer_model.save(f"../../data/monte_outputs/cancer_specific/models/{cancer}_model.pkl")

SKCM: 2815 = 2628 + 187 : True
READ: 2815 = 2652 + 40 : False
LUAD: 2815 = 2478 + 189 : False
BRCA: 2815 = 2501 + 314 : True
COAD: 2815 = 2652 + 123 : False
GBM: 2815 = 2550 + 57 : False
PRAD: 2815 = 2616 + 199 : True
THCA: 2815 = 2614 + 201 : True
LGG: 2815 = 2550 + 208 : False
KIRP: 2815 = 2549 + 110 : False
HNSC: 2815 = 2604 + 211 : True
KIRC: 2815 = 2549 + 130 : False
LUSC: 2815 = 2478 + 148 : False
LIHC: 2815 = 2665 + 150 : True
CESC: 2815 = 2693 + 122 : True
BLCA: 2815 = 2649 + 166 : True
KICH: 2815 = 2549 + 26 : False
UCEC: 2815 = 2617 + 175 : False
ACC: 2815 = 2783 + 32 : True
UCS: 2815 = 2617 + 23 : False


In [ ]:
df_data_model_stats = pd.DataFrame(data_model_stats, columns=["Cancer", "Pan_samples", "Cancer_samples", "Pan_top_n", "Cancer_tau2"])


In [12]:
df_data_model_stats

,Cancer,Pan_samples,Cancer_samples,Pan_top_n,Cancer_tau2
0,SKCM,2628,187,500,100.00
1,READ,2652,40,250,100.00
2,LUAD,2478,189,250,0.10
3,BRCA,2501,314,250,100.00
4,COAD,2652,123,250,100.00
5,GBM,2550,57,500,10.00
6,PRAD,2616,199,250,0.01
7,THCA,2614,201,250,100.00
8,LGG,2550,208,500,10.00
9,KIRP,2549,110,250,100.00


probe correction on test set

In [10]:
for cancer in cancer_types:
    if cancer == "OV":
        continue

    cancer_model = Monte.load(f"../../data/monte_outputs/cancer_specific/models/{cancer}_model.pkl")

    # cancer data
    df_meta_test_cancer = df_meta_test[df_meta_test["Cancer.type"] == cancer]
    df_beta_test_cancer = df_beta_test[df_meta_test["Cancer.type"] == cancer]
    
    df_beta_test_cancer_corrected = cancer_model.purify_values(df_beta_test_cancer, alpha=0.05)
    df_beta_test_cancer_corrected.to_parquet(f"../../data/monte_outputs/cancer_specific/corrected_data/{cancer}_test_beta_corrected.parquet")

Adjusting 57868 probes passing significance threshold of 0.05.
Adjusting 21015 probes passing significance threshold of 0.05.
Adjusting 56358 probes passing significance threshold of 0.05.
Adjusting 73743 probes passing significance threshold of 0.05.
Adjusting 40109 probes passing significance threshold of 0.05.
Adjusting 22386 probes passing significance threshold of 0.05.
Adjusting 37709 probes passing significance threshold of 0.05.
Adjusting 60731 probes passing significance threshold of 0.05.
Adjusting 79964 probes passing significance threshold of 0.05.
Adjusting 39918 probes passing significance threshold of 0.05.
Adjusting 60089 probes passing significance threshold of 0.05.
Adjusting 30870 probes passing significance threshold of 0.05.
Adjusting 50996 probes passing significance threshold of 0.05.
Adjusting 23358 probes passing significance threshold of 0.05.
Adjusting 33865 probes passing significance threshold of 0.05.
Adjusting 42727 probes passing significance threshold o

In [15]:
test_set_size = []
for cancer in cancer_types:
    if cancer == "OV":
        continue
    # cancer data
    df_meta_test_cancer = df_meta_test[df_meta_test["Cancer.type"] == cancer]
    df_beta_test_cancer = df_beta_test[df_meta_test["Cancer.type"] == cancer]
    test_set_size.append(len(df_meta_test_cancer))

In [16]:
df_data_model_stats["Test_samples"] = test_set_size

In [18]:
df_data_model_stats = df_data_model_stats[["Cancer", "Pan_samples", "Cancer_samples", "Test_samples", "Pan_top_n", "Cancer_tau2"]]

In [20]:
df_data_model_stats = df_data_model_stats.sort_values(by="Cancer_samples", ascending=False)

In [22]:
df_data_model_stats.to_csv("../../data/monte_outputs/cancer_specific/stats/data_model_stats.csv", index=False)